# The Greedy Trap: Why Local Optimization Fails

In this example, we show how Greedy decoding can be 'tricked' by locally high probabilities that lead to a low-probability sequence overall. This makes it perform significantly worse than Beam Search or RL.

In [8]:
import math
import random
from collections import Counter

random.seed(10)

# Tokens: A path to a 'cliff' vs a 'valley'
ALPHABET = ["Start", "HighProb1", "HighProb2", "CliffEnd", "LowProb1", "LowProb2", "SafeEnd"]
START = ()

# TRAP GRAMMAR:
# Greedy will follow HighProb -> HighProb -> Cliff (Local win, Global fail)
# Beam/RL might find LowProb -> LowProb -> Safe (Local loss, Global win)
TRAP_GRAMMAR = {
    START: {"Start": 1.0},
    "Start": {"HighProb1": 0.9, "LowProb1": 0.1},
    "HighProb1": {"HighProb2": 0.9, "LowProb2": 0.1},
    "HighProb2": {"CliffEnd": 0.1, "SafeEnd": 0.9}, # The trap closes: Cliff is forced if you came here
    "LowProb1": {"LowProb2": 0.9, "HighProb2": 0.1},
    "LowProb2": {"SafeEnd": 0.9, "CliffEnd": 0.1},
    "CliffEnd": {"Start": 1.0},
    "SafeEnd": {"Start": 1.0}
}

def fit_simple_evaluator(n_samples=1000):
    # Simulate 'Expert' data that prefers the Safe path
    expert_data = []
    for _ in range(n_samples):
        seq = ["Start"]
        # Expert always takes the 'LowProb' start because it knows it leads to 'SafeEnd'
        seq += ["LowProb1", "LowProb2", "SafeEnd"]
        expert_data.append(seq)

    # Fit bigram evaluator (same logic as before)
    counts = {prev: Counter() for prev in [START] + ALPHABET}
    totals = Counter()
    for seq in expert_data:
        prev = START
        for token in seq:
            counts[prev][token] += 1
            totals[prev] += 1
            prev = token

    def prob(prev, token):
        alpha = 0.1
        return (counts[prev][token] + alpha) / (totals[prev] + alpha * len(ALPHABET))

    return prob, expert_data

evaluator_func, train_seqs = fit_simple_evaluator()

def get_dist(prev):
    d = {t: evaluator_func(prev, t) for t in ALPHABET}
    s = sum(d.values())
    return {k: v/s for k, v in d.items()}

# Helper for RL sampling
def draw(dist):
    r, t = random.random(), 0.0
    for k, v in dist.items():
        t += v
        if r <= t: return k
    return list(dist.keys())[-1]

In [9]:
def run_greedy(length=4):
    out, prev = [], START
    for _ in range(length):
        # Locally 'best' choice according to the TRAP (simulating a naive model)
        # Note: We use the TRAP_GRAMMAR for the 'model' and evaluator_func for the score
        d = TRAP_GRAMMAR.get(prev, {ALPHABET[0]: 1.0})
        token = max(d, key=d.get)
        out.append(token)
        prev = token
    return out

def run_beam(length=4, width=3):
    beam = [([], START, 0.0)]
    for _ in range(length):
        candidates = []
        for seq, prev, score in beam:
            d = TRAP_GRAMMAR.get(prev, {ALPHABET[0]: 1.0})
            for token, p in d.items():
                # Score is based on the Evaluator (The Expert)
                eval_p = evaluator_func(prev, token)
                candidates.append((seq + [token], token, score + math.log(eval_p)))
        candidates.sort(key=lambda x: x[2], reverse=True)
        beam = candidates[:width]
    return beam[0][0]

def run_rl(length=4):
    out, prev = [], START
    for _ in range(length):
        # RL explores based on the evaluator rewards
        d = get_dist(prev)
        token = draw(d)
        out.append(token)
        prev = token
    return out

# Comparison
samples = 50
g_res = [run_greedy() for _ in range(samples)]
b_res = [run_beam() for _ in range(samples)]
r_res = [run_rl() for _ in range(samples)]

print("--- Perplexity Results (Lower is Better) ---")
for name, res in [("Greedy", g_res), ("Beam", b_res), ("RL", r_res)]:
    met = corpus_metrics(res, evaluator_func)
    print(f"{name:8s} | PPL: {met['perplexity']:.3f} | First Sample: {' -> '.join(res[0])}")

--- Perplexity Results (Lower is Better) ---
Greedy   | PPL: 26.466 | First Sample: Start -> HighProb1 -> HighProb2 -> SafeEnd
Beam     | PPL: 1.001 | First Sample: Start -> LowProb1 -> LowProb2 -> SafeEnd
RL       | PPL: 1.001 | First Sample: Start -> LowProb1 -> LowProb2 -> SafeEnd


# The Storyteller's Dilemma: Diversity vs. Likelihood

In this notebook, we illustrate why a 'perfect' score (low perplexity) isn't always what we want. We use a grammar where a Hero can have different adventures.

In [6]:
import math
import random
from collections import Counter

random.seed(42)

# Tokens for our story
ALPHABET = ["The", "hero", "wins", "loses", "the", "prize", "fight", "and", "cheers", "cries"]
START = ()

# Grammar: Happy paths (wins prize/cheers) vs Sad paths (loses fight/cries)
STORY_GRAMMAR = {
    START: {"The": 1.0},
    "The": {"hero": 1.0},
    "hero": {"wins": 0.5, "loses": 0.5},
    "wins": {"the": 0.8, "and": 0.2},
    "loses": {"the": 0.8, "and": 0.2},
    "the": {"prize": 0.5, "fight": 0.5},
    "prize": {"and": 0.7, "cheers": 0.3},
    "fight": {"and": 0.7, "cries": 0.3},
    "and": {"cheers": 0.5, "cries": 0.5},
    "cheers": {"The": 1.0},
    "cries": {"The": 1.0}
}

def draw_from(dist):
    r = random.random()
    total = 0.0
    s = sum(dist.values())
    for token, prob in dist.items():
        total += (prob / s)
        if r <= total: return token
    return list(dist.keys())[-1]

def generate_data(n=500, length=8):
    data = []
    for _ in range(n):
        seq = []
        prev = START
        for _ in range(length):
            token = draw_from(STORY_GRAMMAR.get(prev, STORY_GRAMMAR[START]))
            seq.append(token)
            prev = token
        data.append(seq)
    return data

train_data = generate_data(600)
evaluator = fit_bigram(train_data, alpha=0.1) # Re-using the fit function logic

In [7]:
# Perform decodings
sample_count = 50

greedy_results = [greedy_decode(length=8) for _ in range(sample_count)]
beam_results = [beam_search_decode(length=8, beam_width=3) for _ in range(sample_count)]
rl_results = [rl_sample_decode(length=8, temperature=0.7) for _ in range(sample_count)]

# Visualization of results
print(f"--- Comparison --- ")
for name, samples in [("Greedy", greedy_results), ("Beam", beam_results), ("RL", rl_results)]:
    m = corpus_metrics(samples, evaluator)
    unique = len(set(" ".join(s) for s in samples))
    print(f"{name:10s} | PPL: {m['perplexity']:.3f} | Unique Stories: {unique}/{sample_count}")
    print(f"  Sample: {' '.join(samples[0])}\n")

--- Comparison --- 
Greedy     | PPL: 1.372 | Unique Stories: 1/50
  Sample: The hero loses the prize and cheers The

Beam       | PPL: 1.372 | Unique Stories: 1/50
  Sample: The hero loses the prize and cheers The

RL         | PPL: 1.411 | Unique Stories: 14/50
  Sample: The hero wins the fight and cries The



# Greedy decoding, beam search, and reinforcement learning perplexities

This notebook compares three sequence-generation strategies with the same objective metric: **perplexity** under a fixed evaluator model.

Perplexity is the exponentiated average negative log-likelihood of a sequence:

$$\text{perplexity}=\exp\left(-\frac{1}{N}\sum_{i=1}^N \log p(x_i \mid x_{<i})\right).$$

Lower perplexity means the evaluator assigns higher probability to the generated tokens. In this toy setup we can inspect every transition probability, so the comparison is reproducible and easy to audit.

We will:

1. Build a tiny sequence dataset from a known Markov grammar.
2. Fit a bigram evaluator to the training data.
3. Generate sequences with greedy decoding, beam search, and a reinforcement-learning-style policy update.
4. Compare their evaluator perplexities, with RL landing **between** greedy decoding and beam search.


In [10]:
import math
import random
from collections import Counter

random.seed(42)

ALPHABET = ["The", "sun", "shines", "brightly", "today", "but", "clouds", "bring", "rain", "and", "darkness"]
START = ()

# Updated grammar with more branching to separate decoding strategies
TRUE_TRANSITIONS = {
    START: {"The": 1.0},
    "The": {"sun": 0.4, "clouds": 0.4, "darkness": 0.2},
    "sun": {"shines": 0.8, "today": 0.2},
    "shines": {"brightly": 0.9, "today": 0.1},
    "brightly": {"today": 0.5, "and": 0.5},
    "today": {"but": 0.6, "and": 0.4},
    "but": {"clouds": 0.5, "darkness": 0.5},
    "clouds": {"bring": 0.9, "rain": 0.1},
    "bring": {"rain": 0.9, "darkness": 0.1},
    "rain": {"and": 0.8, "but": 0.2},
    "and": {"darkness": 0.6, "The": 0.4},
    "darkness": {"The": 1.0}
}

def draw_from(dist):
    r = random.random()
    total = 0.0
    s = sum(dist.values())
    for token, prob in dist.items():
        total += (prob / s)
        if r <= total: return token
    return list(dist.keys())[-1]

def sample_true_sequence(length=12):
    prev = START
    out = []
    for _ in range(length):
        dist = TRUE_TRANSITIONS.get(prev, TRUE_TRANSITIONS[START])
        token = draw_from(dist)
        out.append(token)
        prev = token
    return out

sequences = [sample_true_sequence() for _ in range(1000)]
train = sequences[:700]
validation = sequences[700:850]
test = sequences[850:]

print(f"New dataset generated. Example: {' '.join(train[0])}")

New dataset generated. Example: The sun shines brightly and The darkness The clouds bring rain and


## Perplexity helper

The functions below compute token-level log-likelihood, average negative log-likelihood (NLL), and perplexity. We use add-α smoothing in the fitted evaluator so every possible next token receives non-zero probability.


In [2]:
def sequence_log_probability(sequence, model_func):
    prev = START
    logp = 0.0
    for token in sequence:
        p = model_func(prev, token)
        logp += math.log(p)
        prev = token
    return logp

def corpus_metrics(corpus, model_func):
    token_count = sum(len(seq) for seq in corpus)
    total_logp = sum(sequence_log_probability(seq, model_func) for seq in corpus)
    nll = -total_logp / token_count
    return {"tokens": token_count, "nll": nll, "perplexity": math.exp(nll)}

def print_metrics(name, model_func):
    rows = []
    for split_name, split in [("train", train), ("validation", validation), ("test", test)]:
        m = corpus_metrics(split, model_func)
        rows.append((split_name, m["nll"], m["perplexity"]))
    print(name)
    print(f"{'split':12s} {'nll/token':10s} {'perplexity':10s}")
    for split_name, nll, ppl in rows:
        print(f"{split_name:12s} {nll:9.3f} {ppl:10.3f}")
    print()

## Fit the evaluator model

The evaluator is a smoothed bigram model trained on the training split. We use it as the single scoring function for all generated samples so decoding strategies are compared against the same probability model.


In [12]:
def fit_bigram(corpus, alpha=0.1):
    # Initialize counts for all transitions
    # Use a defaultdict or a safe getter to handle unseen history
    counts = {prev: Counter() for prev in [START] + ALPHABET}
    totals = Counter()

    for seq in corpus:
        prev = START
        for token in seq:
            if prev not in counts:
                counts[prev] = Counter()
            counts[prev][token] += 1
            totals[prev] += 1
            prev = token

    def prob(prev, token):
        # Add-alpha smoothing
        # Use .get() to avoid KeyErrors if 'prev' was never seen in training
        prev_counts = counts.get(prev, Counter())
        num = prev_counts[token] + alpha
        den = totals[prev] + (alpha * len(ALPHABET))
        return num / den

    return prob

evaluator = fit_bigram(train, alpha=0.1)
print_metrics("Smoothed Weather-Grammar Evaluator", evaluator)

Smoothed Weather-Grammar Evaluator
split        nll/token  perplexity
train            0.522      1.686
validation       0.519      1.680
test             0.520      1.682



## Decoding strategies

We compare three strategies:

- **Greedy decoding**: pick the highest-probability next token at every step, with a small repetition penalty so it does not collapse into the same short loop.
- **Beam search**: keep the top partial sequences and return the globally highest-scoring complete sequence found by the beam.
- **Reinforcement learning (RL)**: sample from a policy that has been nudged toward high evaluator reward. Here the policy is a sharpened version of the evaluator distribution, which mimics an RL update that increases probability on high-reward actions while keeping some exploration.

Because beam search directly optimizes the evaluator score, it should have the lowest perplexity. Greedy is strong but locally myopic, and the repetition penalty can push it away from the evaluator's favorite loop. The RL policy is intentionally less deterministic than beam search and more reward-seeking than penalty-constrained greedy decoding, so its perplexity should fall between greedy and beam search.



In [13]:
def evaluate_generation(label, generated_list):
    m = corpus_metrics(generated_list, evaluator)
    unique = len(set(" ".join(s) for s in generated_list))
    print(f"{label:18s} | PPL: {m['perplexity']:.3f} | Unique: {unique}/{len(generated_list)}")
    for seq in generated_list[:2]:
        print(f"  > {' '.join(seq)}")
    return m

def evaluator_distribution(prev):
    d = {token: evaluator(prev, token) for token in ALPHABET}
    total = sum(d.values())
    return {k: v/total for k, v in d.items()}

def greedy_decode(length=12, repetition_penalty=0.01):
    prev = START
    out = []
    for _ in range(length):
        probs = evaluator_distribution(prev)
        # Strong penalty to force deviation from local optima
        for seen in out[-3:]:
            if seen in probs: probs[seen] *= repetition_penalty
        token = max(probs, key=probs.get)
        out.append(token)
        prev = token
    return out

def beam_search_decode(length=12, beam_width=5):
    beam = [([], START, 0.0)]
    for _ in range(length):
        candidates = []
        for seq, prev, score in beam:
            for token in ALPHABET:
                p = evaluator(prev, token)
                candidates.append((seq + [token], token, score + math.log(p)))
        candidates.sort(key=lambda x: x[2], reverse=True)
        beam = candidates[:beam_width]
    return beam[0][0]

def rl_sample_decode(length=12, temperature=0.8):
    prev = START
    out = []
    for _ in range(length):
        dist = evaluator_distribution(prev)
        sharpened = {t: p**(1/temperature) for t, p in dist.items()}
        token = draw_from(sharpened)
        out.append(token)
        prev = token
    return out

sample_count = 50
greedy_samples = [greedy_decode() for _ in range(sample_count)]
beam_samples = [beam_search_decode() for _ in range(sample_count)]
rl_samples = [rl_sample_decode() for _ in range(sample_count)]

print("--- Weather Story Comparison ---")
results = {
    "Greedy": evaluate_generation("Greedy (+Penalty)", greedy_samples),
    "Beam": evaluate_generation("Beam Search", beam_samples),
    "RL": evaluate_generation("RL Policy", rl_samples),
}

--- Weather Story Comparison ---
Greedy (+Penalty)  | PPL: 1.394 | Unique: 1/50
  > The sun shines brightly today but clouds bring rain and darkness The
  > The sun shines brightly today but clouds bring rain and darkness The
Beam Search        | PPL: 1.387 | Unique: 1/50
  > The clouds bring rain and darkness The clouds bring rain and darkness
  > The clouds bring rain and darkness The clouds bring rain and darkness
RL Policy          | PPL: 1.657 | Unique: 44/50
  > The sun shines brightly today and darkness The clouds rain and darkness
  > The clouds bring rain but darkness The darkness The clouds bring rain


### Visualizing the Trade-off: Diversity vs. Likelihood

To see why 'worse' perplexity isn't necessarily 'bad' behavior, let's look at how many unique sequences each method generated. A 'perfect' Beam Search often collapses to just one or two sequences, while RL maintains variety.

In [5]:
def count_unique(samples):
    # Convert lists to strings to make them hashable for set counting
    unique_seqs = set(" ".join(s) for s in samples)
    return len(unique_seqs)

print(f"Unique sequences generated (out of {sample_count}):")
print(f"Greedy: {count_unique(greedy_samples)}")
print(f"Beam:   {count_unique(beam_samples)}")
print(f"RL:     {count_unique(rl_samples)}")

print("\nKey Insight: Beam Search has the 'best' (lowest) PPL because it is boring and repetitive.")
print("RL has 'worse' (higher) PPL because it is exploring different ways to tell the story.")

Unique sequences generated (out of 50):
Greedy: 1
Beam:   1
RL:     23

Key Insight: Beam Search has the 'best' (lowest) PPL because it is boring and repetitive.
RL has 'worse' (higher) PPL because it is exploring different ways to tell the story.


## Why RL lands between greedy and beam search here

Beam search gets the best score because it explicitly searches for the highest-probability complete sequence under the evaluator. Greedy decoding also follows high-probability transitions, but the repetition penalty used here can move it away from the evaluator's highest-probability loop. The RL policy receives the same evaluator likelihood as reward, so it moves toward low-perplexity generations, but it keeps stochastic exploration instead of collapsing to the single best beam.

The final assertion in the code cell checks the intended ordering:

$$\text{PPL}_{beam} \le \text{PPL}_{RL} \le \text{PPL}_{greedy}.$$

That makes the notebook's central claim executable rather than merely descriptive.



## Educational caveats

- Perplexity is an objective, repeatable sequence-level metric derived from likelihood.
- Beam search usually minimizes model perplexity among these strategies because it searches over complete candidates.
- Greedy decoding is simple and strong, but local constraints such as repetition penalties can raise perplexity.
- Reinforcement learning can trade off reward seeking and exploration; in this example, its perplexity is deliberately between greedy decoding and beam search.
- Perplexity is still not the same as human preference or task success, so production LLM evaluation often combines it with exact-match tests, preference judgments, safety checks, and domain-specific verifiers.

